In [12]:
!pip install pandas
!pip install numpy

In [13]:
import pandas as pd
import numpy as np

path = r'C:\Users\maxle\UCInsure\src\hurricane_model\NRI_Table_CensusTracts.csv'
df = pd.read_csv(path, low_memory=False)



  Using cached tzdata-2025.3-py2.py3-none-any.whl.metadata (1.4 kB)
   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   -------------------------- ------------- 6.6/9.9 MB 55.9 MB/s eta 0:00:01
   ---------------------------------------- 9.9/9.9 MB 57.9 MB/s  0:00:00
   ---------------------------------------- 0.0/12.4 MB ? eta -:--:--
   ---------------------------------------- 12.4/12.4 MB 79.4 MB/s  0:00:00
Using cached tzdata-2025.3-py2.py3-none-any.whl (348 kB)

   ---------------------------------------- 0/3 [tzdata]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   

Column selection extracting relevent columns

In [14]:
hurricane_cols = [
    # --- Frequency ---
    'HRCN_EVNTS',       # Total hurricane events recorded in this tract
    'HRCN_AFREQ',       # Average annual hurricane frequency
 
    # --- Exposure ---
    'HRCN_EXP_AREA',    # Land area exposed to hurricane hazard (sq km)
    'HRCN_EXPB',        # Building value exposed to hurricane hazard (USD)
    'HRCN_EXPP',        # Population exposed to hurricane hazard
 
    # --- Historic loss ---
    'HRCN_HLRB',        # Historic loss ratio — buildings (fraction lost)
    'HRCN_HLRP',        # Historic loss ratio — population
 
    # --- Tract characteristics ---
    'BUILDVALUE',       # Total building replacement value in tract (USD)
    'POPULATION',       # Total tract population
    'AREA',             # Tract area (sq km)
 
    # --- Social/resilience context ---
    'SOVI_SCORE',       # Social vulnerability index
    'RESL_SCORE',       # Community resilience score
]


In [16]:
LABEL_COL  = 'HRCN_EALB'     # Expected Annual Loss — Buildings (USD) ← training target
ID_COLS    = ['TRACTFIPS', 'STATE', 'COUNTY', 'TRACT']
 
ALL_COLS = ID_COLS + hurricane_cols + [LABEL_COL]
df_hurricane = df[ALL_COLS].copy()
 
print(f"Loaded {len(df_hurricane):,} census tracts.")
print(f"Columns: {len(hurricane_cols)} features + 1 label\n")
 



Loaded 85,154 census tracts.
Columns: 12 features + 1 label



In [17]:
COASTAL_STATES = [
    'Florida', 'Texas', 'Louisiana', 'Mississippi', 'Alabama', 'Georgia',
    'South Carolina', 'North Carolina', 'Virginia', 'Maryland', 'Delaware',
    'New Jersey', 'New York', 'Connecticut', 'Rhode Island', 'Massachusetts',
    'New Hampshire', 'Maine'
]


df_hurricane = df_hurricane[df_hurricane['STATE'].isin(COASTAL_STATES)].copy()
print(f"After coastal state filter: {len(df_hurricane):,} tracts remaining.")



After coastal state filter: 37,415 tracts remaining.


In [18]:
before = len(df_hurricane)
df_hurricane = df_hurricane[
    df_hurricane[LABEL_COL].notna() &
    (df_hurricane[LABEL_COL] > 0)
]
print(f"After removing null/zero labels: {len(df_hurricane):,} tracts "
      f"({before - len(df_hurricane):,} dropped).\n")



After removing null/zero labels: 37,288 tracts (127 dropped).



In [20]:
missing_report = df_hurricane[hurricane_cols].isna().sum()
missing_report = missing_report[missing_report > 0]
if not missing_report.empty:
    print("Missing values per feature:")
    print(missing_report.to_string())
    print()
 
df_hurricane[hurricane_cols] = df_hurricane[hurricane_cols].fillna(0)



In [22]:
df_hurricane['POP_DENSITY'] = np.where(
    df_hurricane['AREA'] > 0,
    df_hurricane['POPULATION'] / df_hurricane['AREA'],
    0
)
 
# Exposure ratio: what fraction of total building value is exposed to hurricanes
df_hurricane['EXPOSURE_RATIO'] = np.where(
    df_hurricane['BUILDVALUE'] > 0,
    df_hurricane['HRCN_EXPB'] / df_hurricane['BUILDVALUE'],
    0
).clip(0, 1)
 
ENGINEERED_COLS = ['POP_DENSITY', 'EXPOSURE_RATIO']
ALL_FEATURE_COLS = hurricane_cols + ENGINEERED_COLS
 
print(f"Engineered features added: {ENGINEERED_COLS}\n")

Engineered features added: ['POP_DENSITY', 'EXPOSURE_RATIO']



In [23]:
log_eal = np.log1p(df_hurricane[LABEL_COL])   # log1p handles zero safely
df_hurricane['LABEL_NORM'] = log_eal / log_eal.max()   # [0, 1]
 
print("Label distribution (HRCN_EALB in USD):")
print(df_hurricane[LABEL_COL].describe().apply(lambda x: f"${x:,.0f}").to_string())
print()
print("Normalised label (0–1):")
print(df_hurricane['LABEL_NORM'].describe().round(4).to_string())

Label distribution (HRCN_EALB in USD):
count        $37,288
mean        $298,027
std         $676,907
min               $1
25%          $15,202
50%          $56,924
75%         $262,367
max      $15,760,922

Normalised label (0–1):
count    37288.0000
mean         0.6569
std          0.1289
min          0.0434
25%          0.5810
50%          0.6607
75%          0.7529
max          1.0000


In [24]:
output_cols = ID_COLS + ALL_FEATURE_COLS + [LABEL_COL, 'LABEL_NORM']
output_path = r'C:\Users\maxle\UCInsure\src\hurricane_model\hurricane_training_data.csv'
 
df_hurricane[output_cols].to_csv(output_path, index=False)
print(f"\nSaved {len(df_hurricane):,} rows → {output_path}")
print(f"Final shape: {len(df_hurricane)} rows × {len(ALL_FEATURE_COLS)} features")


Saved 37,288 rows → C:\Users\maxle\UCInsure\src\hurricane_model\hurricane_training_data.csv
Final shape: 37288 rows × 14 features
